# Huawei-Inspired Healthcare AI Lab 6

> Streamlined GitHub/Colab edition. Full commented edition is available in the course Google Drive folder.

In [ ]:
!pip -q install ucimlrepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('PyTorch version:', torch.__version__)
print('Packages imported successfully.')

In [ ]:
random_tensor = torch.rand(4, 3)
print(random_tensor)

In [ ]:
temperature = torch.tensor(37.2)
one_record = torch.tensor([58.0, 82.0, 135.0])
three_records = torch.tensor([[58.0, 82.0, 135.0], [42.0, 75.0, 120.0], [67.0, 91.0, 145.0]])
print('Scalar:', temperature)
print('Vector:', one_record)
print('Matrix:')
print(three_records)

In [ ]:
numpy_values = np.array([[92.0, 98.0], [110.0, 125.0], [145.0, 160.0]], dtype=np.float32)
glucose_tensor = torch.from_numpy(numpy_values)
print(glucose_tensor)
print('Tensor data type:', glucose_tensor.dtype)

In [ ]:
print('Shape:', three_records.shape)
print('Number of dimensions:', three_records.ndim)
print('Number of stored values:', three_records.numel())
print('Data type:', three_records.dtype)
print('Device:', three_records.device)

In [ ]:
zeros_tensor = torch.zeros(2, 3)
ones_tensor = torch.ones(2, 3)
range_tensor = torch.arange(6).reshape(2, 3)
print('Zeros:')
print(zeros_tensor)
print('\nOnes:')
print(ones_tensor)
print('\nRange:')
print(range_tensor)

In [ ]:
print('Complete table:')
print(three_records)
print('\nFirst record:')
print(three_records[0])
print('\nHeart-rate column:')
print(three_records[:, 1])
print('\nOne selected value:')
print(three_records[2, 2])

In [ ]:
original_tensor = torch.arange(6)
reshaped_tensor = original_tensor.reshape(2, 3)
flattened_tensor = reshaped_tensor.flatten()
print('Original vector:', original_tensor)
print('Reshaped table:')
print(reshaped_tensor)
print('Flattened vector:', flattened_tensor)

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print('Addition:', a + b)
print('Element-by-element multiplication:', a * b)
print('Dot product:', torch.dot(a, b))
print('Mean of a:', a.mean())
print('Maximum of b:', b.max())

In [ ]:
first_group = torch.tensor([[50.0, 80.0], [60.0, 85.0]])
second_group = torch.tensor([[70.0, 90.0]])
combined_groups = torch.cat([first_group, second_group], dim=0)
print(combined_groups)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Selected device:', device)
tensor_on_device = three_records.to(device)
print('Tensor device:', tensor_on_device.device)

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y_value = x ** 2
y_value.backward()
print('x:', x.item())
print('y = x²:', y_value.item())
print('Gradient of y with respect to x:', x.grad.item())

In [ ]:
dataset = fetch_ucirepo(id=519)
feature_table = dataset.data.features.copy()
target_table = dataset.data.targets.copy()
print('Dataset name:', dataset.metadata.name)
print('Feature-table shape:', feature_table.shape)
print('Target-table shape:', target_table.shape)
feature_table.head()

In [ ]:
feature_table = feature_table.apply(pd.to_numeric, errors='coerce')
target_series = pd.to_numeric(target_table.iloc[:, 0], errors='coerce')
complete_table = feature_table.copy()
complete_table['target'] = target_series
complete_table = complete_table.dropna()
X = complete_table.drop(columns='target')
y = complete_table['target'].astype(int)
print('Usable records:', len(X))
print('Number of input features:', X.shape[1])
print('Target values:', sorted(y.unique().tolist()))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Training records:', len(X_train))
print('Testing records:', len(X_test))

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('First standardised training record:')
print(X_train_scaled[0])

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)
print('Training-feature tensor shape:', X_train_tensor.shape)
print('Training-label tensor shape:', y_train_tensor.shape)
print('Input data type:', X_train_tensor.dtype)
print('Label data type:', y_train_tensor.dtype)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
print('Training dataset length:', len(train_dataset))
print('Testing dataset length:', len(test_dataset))
first_features, first_label = train_dataset[0]
print('First feature tensor shape:', first_features.shape)
print('First label:', first_label.item())

In [ ]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print('Number of training batches:', len(train_loader))
print('Number of testing batches:', len(test_loader))

In [ ]:
batch_features, batch_labels = next(iter(train_loader))
print('Feature-batch shape:', batch_features.shape)
print('Label-batch shape:', batch_labels.shape)
print('First three labels:', batch_labels[:3])
print('First record in the batch:')
print(batch_features[0])

In [ ]:
batch_features = batch_features.to(device)
batch_labels = batch_labels.to(device)
print('Feature-batch device:', batch_features.device)
print('Label-batch device:', batch_labels.device)

In [ ]:
training_numpy = X_train_tensor.cpu().numpy()
feature_names = list(X.columns)
ejection_index = feature_names.index('ejection_fraction')
plt.figure(figsize=(8, 5))
plt.hist(training_numpy[:, ejection_index], bins=15, edgecolor='black')
plt.xlabel('Standardised ejection fraction')
plt.ylabel('Number of training records')
plt.title('One Healthcare Feature Stored in a Tensor')
plt.show()

In [ ]:
tensor_file_name = 'heart_failure_prepared_tensors.pt'
torch.save({'X_train': X_train_tensor, 'y_train': y_train_tensor, 'X_test': X_test_tensor, 'y_test': y_test_tensor, 'feature_names': feature_names}, tensor_file_name)
print('Saved:', tensor_file_name)

In [ ]:
loaded_data = torch.load(tensor_file_name, map_location='cpu')
print('Loaded items:', list(loaded_data.keys()))
print('Loaded training shape:', loaded_data['X_train'].shape)
print('Loaded feature names:', loaded_data['feature_names'])